In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
import numpy as np

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV


In [2]:
df_train = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/train_data.csv", encoding = "utf-8")
df_val = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/val_data.csv", encoding = "utf-8")

df_train['Datum'] = pd.to_datetime(df_train['Datum'], format='%Y-%m-%d')
df_val['Datum'] = pd.to_datetime(df_val['Datum'], format='%Y-%m-%d')

In [3]:
df_train.head()

,id,Datum,Warengruppe,Umsatz,KielerWoche,Bewoelkung,Temperatur,Windgeschwindigkeit,Woche,Monat,...,Ferien,sunny,cloudy,rainy,thunderstorm,is_weekend,sin_Monat,cos_Monat,sin_Wochentag,cos_Wochentag
0,1307011,2013-07-01,1,148.828353,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
1,1307012,2013-07-01,2,535.856285,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
2,1307013,2013-07-01,3,201.198426,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
3,1307014,2013-07-01,4,65.890169,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349
4,1307015,2013-07-01,5,317.475875,0,6,17.8375,15,27,7,...,0,0,1,0,0,0,-0.5,-0.866025,0.781831,0.62349


In [4]:
# alvo
TARGET = 'Umsatz'

# colunas que NÃO vão para o modelo
cols_drop = ['id', TARGET, 'Datum']

X_train = df_train.drop(columns=cols_drop)
y_train = df_train[TARGET]
X_val = df_val.drop(columns=cols_drop)
y_val = df_val[TARGET]

In [5]:
#check na values y_train
print(y_train.isnull().sum())

0


In [6]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)


,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
# previsões
y_pred_train = rf.predict(X_train)
y_pred_val = rf.predict(X_val)

def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100


# métricas treino
mae_tr = mean_absolute_error(y_train, y_pred_train)
r2_tr = r2_score(y_train, y_pred_train)
mape_tr = mape_safe(y_train, y_pred_train)

# métricas validação
mae_val = mean_absolute_error(y_val, y_pred_val)
r2_val = r2_score(y_val, y_pred_val)
mape_val = mape_safe(y_val, y_pred_val)

print('--- TREINO ---')
print(f'MAE  : {mae_tr:,.2f}')
print(f'R²   : {r2_tr:,.3f}')
print(f'MAPE : {mape_tr:,.2f}%')

print('\n--- VALIDAÇÃO ---')
print(f'MAE  : {mae_val:,.2f}')
print(f'R²   : {r2_val:,.3f}')
print(f'MAPE : {mape_val:,.2f}%')

--- TREINO ---
MAE  : 11.76
R²   : 0.980
MAPE : 6.67%

--- VALIDAÇÃO ---
MAE  : 35.51
R²   : 0.818
MAPE : 20.84%


In [8]:
from sklearn.model_selection import ParameterSampler

param_dist = {
    "n_estimators": [300, 600, 900, 1200],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50],
    "min_samples_split": [2, 5, 10, 20],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 1.0],
    "bootstrap": [True, False],
}

best_mape = np.inf
best_params = None
best_model = None

for params in ParameterSampler(param_dist, n_iter=60, random_state=42):
    model = RandomForestRegressor(**params, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    score = mape_safe(y_val, pred_val)

    if score < best_mape:
        best_mape = score
        best_params = params
        best_model = model

print("Best VAL MAPE:", best_mape)
print("Best params:", best_params)


KeyboardInterrupt: 

In [ ]:
best_params = {
    'n_estimators': 600, 
    'min_samples_split': 20, 
    'min_samples_leaf': 2, 
    'max_features': 1.0, 
    'max_depth': 20, 
    'bootstrap': True,
    'max_depth': 20, 
    'bootstrap': True
}


In [ ]:
y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

print('--- TREINO ---')
print(f"MAE  : {mean_absolute_error(y_train, y_pred_train):,.2f}")
print(f"R²   : {r2_score(y_train, y_pred_train):,.3f}")
print(f"MAPE : {mape_safe(y_train, y_pred_train):,.2f}%")

print('\n--- VALIDAÇÃO ---')
print(f"MAE  : {mean_absolute_error(y_val, y_pred_val):,.2f}")
print(f"R²   : {r2_score(y_val, y_pred_val):,.3f}")
print(f"MAPE : {mape_safe(y_val, y_pred_val):,.2f}%")


--- TREINO ---
MAE  : 24.86
R²   : 0.901
MAPE : 13.92%

--- VALIDAÇÃO ---
MAE  : 34.38
R²   : 0.837
MAPE : 20.23%


In [ ]:
df_val_eval = df_val.copy()
df_val_eval["y_true"] = y_val.to_numpy()
df_val_eval["y_pred"] = y_pred_val

mape_by_group = (
    df_val_eval
    .groupby("Warengruppe")
    .apply(lambda x: mape_safe(x["y_true"], x["y_pred"]))
    .reset_index(name="MAPE")
    .sort_values("Warengruppe")
)

print(mape_by_group)
print("Macro-MAPE:", mape_by_group["MAPE"].mean())
print("Micro-MAPE:", mape_safe(df_val_eval["y_true"], df_val_eval["y_pred"]))


   Warengruppe       MAPE
0            1  19.907234
1            2  16.292175
2            3  19.681718
3            4  24.553163
4            5  17.916015
5            6  38.147771
Macro-MAPE: 22.74967951424482
Micro-MAPE: 20.232120765597696


/tmp/ipykernel_56070/1735562070.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: mape_safe(x["y_true"], x["y_pred"]))


In [ ]:
# mesmas colunas que você removeu no treino
TARGET = "Umsatz"
cols_drop = ["id", TARGET, "Datum"]

df_test = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/test_data.csv", encoding="utf-8")
df_test["Datum"] = pd.to_datetime(df_test["Datum"], format="%Y-%m-%d")

X_test = df_test.drop(columns=cols_drop)

# (opcional, mas recomendado) garantir MESMA ordem de colunas do treino
X_test = X_test.reindex(columns=X_train.columns)

y_pred_test = best_model.predict(X_test)

df_test["Umsatz_Predicted"] = y_pred_test



In [ ]:
X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

final_rf = RandomForestRegressor(**best_params, n_jobs=-1, random_state=42)
final_rf.fit(X_trainval, y_trainval)

y_pred_test = final_rf.predict(X_test)

df_test['Umsatz_Predicted'] = y_pred_test
df_test[['id', 'Umsatz_Predicted']].to_csv(
    '/workspaces/bakery_prediction/2_BaselineModel/02_RF/predictions/03_rf_cylindrical_predictions.csv',
    index=False
)


In [ ]:
imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.head(30))
print("\nImportância total das Warengruppe_*:", imp[imp.index.str.startswith("Warengruppe_")].sum())
print("\nImportâncias Warengruppe_*:\n", imp[imp.index.str.startswith("Warengruppe_")])